# Simulations with fixed attractant source

Packages used:

In [ ]:
using Pkg
using CellBasedModels 
using GeometryBasics
using Distributions
using GLMakie, Colors
using CSV, DataFrames, Statistics
using Printf, JLD2
using SpecialFunctions
using LsqFit
using LinearAlgebra
using StatsBase

## Model definition

The following code generates the model, with the rules for the agents, and defines the medium.

In [ ]:
fixed_source = ABM(2,
    agent = Dict(
        :vx => Float64,
        :vy => Float64,
        :v => Float64, 
        :theta => Float64,
        :d => Float64,
        :l => Float64,
        :m => Float64,
        :active => Bool,

        :methyl => Float64,
        :Yp => Float64,
        :G => Float64,
        :λ => Float64,
        :P => Float64,
        :M => Float64,
        :F => Float64,
        :A => Float64
        
    ),

    model = Dict(

        :Dr_run => Float64,

        :ε0 => Float64, 
        :ε1 => Float64,
        :ε2 => Float64,
        :ε3 => Float64,
        :K => Float64,
        :Nrec => Float64, 
        :Ki => Float64,
        :Ka => Float64,
        :τm => Float64, 
        :α => Float64,     
        :ωFrec => Float64,   
        :Ky => Float64,         
        :Z => Float64,         
        :Kz => Float64,        
        :Yy => Float64,        

        :DMedium => Float64,
        :delta => Float64,
    ),

    agentODE = quote  

        r = sqrt((x)^2 + (y)^2)
        ll = sqrt(DMedium / delta)
        r_eff = max(r, 1e-3)

        mm = 1.0 * besselk(0, r_eff / ll)

        F = ε0 + ε1 * methyl + Nrec * log((1 + mm / Ki) / (1 + mm / Ka)) 
        F0 = log(((Ky * (α - K)) / (K * (Kz * Z + Yy))) - 1)       

        mx = (ε0 + Nrec * log((1 + mm / Ki) / (1 + mm / Ka)) - F0) / (- ε1)

        A = 1 / (1 + exp(F))    

        Yp = (Ky * A * α) / ((Ky * A) + (Kz * Z) + Yy)

        G = ε2 / 4 - (ε3 / 2) / (1 + (K / Yp))      

        dt(x) = vx 
        dt(y) = vy  
        dt(methyl) = -(1 / τm) * (methyl - mx)        
        
    end,

    agentRule = quote

        v_run = v
        v_tumble = 0.25 

        speed = active ? v_run : v_tumble

        Dr_tumble = 6.2      
        Dr_total = active ? Dr_run : Dr_tumble

        if active 
            λ = ωFrec*exp(-G)
            P = 1 - exp(-λ * dt)
                
        else
            λ = ωFrec*exp(G)
            P = 1 - exp(-λ * dt)                  
        end


        if active 
            λrt = ωFrec*exp(-G) 

            P_rt = 1 - exp(-λrt * dt)
            P = rand() 
                                                
            if P < P_rt            
                active = false
                vx = speed * cos(theta)
                vy = speed * sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn()           
            else     #Si rate baixa 
                active = true
                vx = speed * cos(theta)
                vy = speed * sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn()       
            end

        else
            λtr = ωFrec*exp(G) 
            P_tr = 1 - exp(-λtr * dt)
            P = rand()

            if P < P_tr
                active = true
                vx = speed * cos(theta)
                vy = speed * sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn()
            else
                active = false
                vx = speed * cos(theta)
                vy = speed * sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn()
            end
        end
    end,
    
    agentAlg = CBMIntegrators.Heun(),
    mediumAlg=DifferentialEquations.Euler(),
    neighborsAlg=CBMNeighbors.CellLinked(cellEdge=10)
)

Then, the community is loaded with the different sets of parameters studied and saved in a .jld2 file. 

In [ ]:
ns = [0.25, 0.5, 0.75, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
steps = 120000
velocities = [5.0, 10.0, 20.0, 100.0]
adaptation = [1.0, 2.0, 4.0, 10.0, 30.0]


for n in ns
    for (idx, v) in enumerate(velocities)
        for (idx_t, tm) in enumerate(adaptation)
            println("Running n=$n")

            com = Community(
                fixed_source,
                N=1000,
                dt=0.01,
                simBox = [-100.0 100.0; -100.0 100.0],
                NMedium = [100, 100]
            )   

            m = 1/100
            g = 1/10000
            d = 1

            com.Dr_run = 0.062

            com.v = v 

            com.ωFrec = 1.3
            com.Ki = 0.0182
            com.Ka = 3.0
            com.Nrec = 6.0
            com.ε0   = 6.0
            com.ε1   = -1.0
            com.ε2   = 80
            com.ε3   = 80

            com.τm = tm

            com.α   = 6.0

            com.K = 2.0 

            com.Ky = 100.0
            com.Kz = 10.0
            com.Z = 5.0
            com.Yy = 0.1

            com.m = 1.        
            com.d = 1.        
            com.l = 3;

            Dc = 10 
            delta = 0.01

            d = delta/(((tm)^2)*(v/10)^2)

            com.DMedium = Dc / n
            com.delta = d * n

            com.x = 2.5*sqrt(com.DMedium/com.delta)
            com.y = 2.5*sqrt(com.DMedium/com.delta)
            com.theta = rand(Uniform(0,2π),com.N)

            com.methyl .= 0.0
            com.Yp .= com.K


            loadToPlatform!(com, preallocateAgents=1000)
            
            outfile = @sprintf("fsa_%s_v_%s_n_%s.jld2", tm, v, n)

            jldopen(outfile, "w") do file

                meta = JLD2.Group(file, "meta")
                meta["DMedium"] = com.DMedium
                meta["delta"] = com.delta

                for step in 1:steps
                    step!(com)

                    if step%100 == 0
                        stepname = @sprintf("step_%06d", step)
                        g = JLD2.Group(file, stepname)

                        # Agent-level arrays (length = N)
                        g["x"] = copy(com.x)
                        g["y"] = copy(com.y)
                        g["theta"] = copy(com.theta)
        
                end
            end
        end
    end
end

## Analysis

First, plot the trajectories to the source of all agents and record the descent time and distance of the agents that capture the source. 

In [ ]:
ns = [0.25, 0.5, 0.75, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
velocities = [5.0, 10.0, 20.0, 100.0]
adaptation = [1.0, 2.0, 4.0, 10.0, 30.0]

steps = 120000

for (idx_v, v) in enumerate(velocities)
    for (idx_t, tm) in enumerate(adaptation)
        rows = 1
        cols = 4
        fig = Figure(size=(900 * cols, 600 * rows))

        for (idx, lambda) in enumerate(ns)
        
            filename = @sprintf("fsa_%s_v_%s_n_%s.jld2", tm, v, lambda)
            
            row = ceil(Int, idx / cols)
            col = (idx - 1) % cols + 1
            
            ax = Axis(fig[row, col], 
                    xlabel="Time (s)", 
                    ylabel="Distance to source (X)",
                    title="Λ = $lambda")

            jldopen(filename, "r") do file 
                for agent in 1:100
                    mean_dist_x = Float64[] 
                    mean_dist_y = Float64[] 

                    for step in 100:100:steps
            
                        g = file[@sprintf("step_%06d", step)]

                        other_x = g["x"][agent]
                        other_y = g["y"][agent]
                        dists_x = other_x .- 0
                        dists_y = other_y .- 0
                        dists = sqrt(dists_x .^2 .+ dists_y .^2)

                        push!(mean_dist, dists)
        
                    end

                    lines!(ax, (100:100:steps)*0.01, mean_dist_x, label="Mean X")
                end
            end
        end
        save("Plots/Trajectories_tm$(tm)_v$(v).pdf", fig)
    end
end

Save the observed time at which the agents that capture the source achieve equilibrium and the threshold distance at which they oscilate. 

Second, save this data in a data frame that will contain: the adaptation time, the velocity, the lambda value, the time at which equilibrium is achieved, and the threshold distance. This will then be used to separate both populations with the following code. 

In [ ]:
function check_agent(file, step, agent_idx, threshold)

    data = file[@sprintf("step_%06d", step)]
    
    x = data["x"][agent_idx]
    y = data["y"][agent_idx]
    
    return (x <= threshold) && (y <= threshold) && (x >= -threshold) && (y >= -threshold)
end

In [ ]:
dfs = Dict{Tuple{Float64, Float64, Float64}, DataFrame}()

ns = [0.25, 0.5, 0.75, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
velocities = [5.0, 10.0, 20.0, 100.0]
adaptation = [1.0, 2.0, 4.0, 10.0, 30.0]

N_agents = 1000
end_step = 120000

for (v_idx, v) in enumerate(velocities)
    for (t_idx, tm) in enumerate(adaptation)
        for (l_idx, lambda) in enumerate(ns)

            filter_1 = dc_delta_all[dc_delta_all.tm .== tm, :]
            filter_2 = filter_1[filter_1.v .== v, :]
            row = filter_2[filter_2.n .== lambda, :]

            start_step = row.eq[1]

            threshold_distance = row.threshold[1]

            # -----------------------------
            # STEP 1: find valid agents
            # -----------------------------
            valid_agents = Int[]

            jldopen(filename, "r") do file
                for agent_idx in 1:N_agents
                    is_valid = true

                    for step in start_step:100:end_step
                        if !check_agent(file, step, agent_idx, threshold_distance)
                            is_valid = false
                            break
                        end
                    end

                    if is_valid
                        push!(valid_agents, agent_idx)
                    end
                end
            end

            println("  ✔ valid agents: $(length(valid_agents))")

            # -----------------------------
            # STEP 2: extract trajectories
            # -----------------------------
            steps_vec = Int[]
            agents_vec = Int[]
            x_vec = Float64[]
            y_vec = Float64[]
            
            jldopen(filename, "r") do file
                for agent in valid_agents
                    for step in 100:100:end_step
                        g = file[@sprintf("step_%06d", step)]

                        push!(steps_vec, step)
                        push!(agents_vec, agent)
                        push!(x_vec, g["x"][agent])
                        push!(y_vec, g["y"][agent])
                    end
                end
            end

            df = DataFrame(
                step = steps_vec,
                agent = agents_vec,
                x = x_vec,
                y = y_vec
            )

            dfs[(tm, v, lambda)] = df

        end
    end
end

Now, with the agents that captured the source, the chemotactic strength (k) and the stochastic fluctuations at equilibrium (D) will be calculated as follows: 

In [ ]:
metrics = DataFrame(
    tm = Float64[],
    v = Float64[],
    l = Float64[],
    sigma = Float64[],
    variance = Float64[],
    k = Float64[],
    D = Float64[],
    P_esc = Float64[]
)


for (idx, l) in enumerate(lambdas)
    for (t_idx, tm) in enumerate(tms)
        for (v_idx, v) in enumerate(vel)

            df = dfs_tm_30[(tm, v, l)]

            filter_1 = dc_delta_all[dc_delta_all.tm .== tm, :]
            filter_2 = filter_1[filter_1.v .== v, :]
            row = filter_2[filter_2.n .== l, :]

            if nrow(row) == 0
                continue
            end

            eq_step = row.eq[1]
            
            if ismissing(eq_step)
                println("Skipping $tm, $l (missing params)")
                continue
            end

            # =========================
            # SPLIT DATA
            # =========================
            eq = df[df.step .>= eq_step, :]
            
            # =========================
            # EQUILIBRIUM STATS
            # =========================
            sigma_x = std(eq.x)
            sigma_y = std(eq.y)

            sigma = (sigma_x + sigma_y) / 2

            variance_x = var(eq.x)
            variance_y = var(eq.y)

            variance = (variance_x + variance_y) / 2

            len_data = length(unique(eq.step))
            lag_data = len_data - 100
            lags = (1:1:lag_data)
            
            grouped_ag = groupby(eq, :agent)
            correlations = zeros(length(grouped_ag), length(lags))
            correlations_y = zeros(length(grouped_ag), length(lags))

            # Calculate escape probability
            P_esc = 1 - (length(grouped_ag) / 1000)

            for i in 1:length(grouped_ag)
                df_ag = grouped_ag[i]
                traj = df_ag.x
                traj_y = df_ag.y
                cor = autocor(traj, lags; demean = false)
                cor_y = autocor(traj_y, lags; demean = false)
                correlations[i, :] = cor
                correlations_y[i, :] = cor_y
            end

            correlations_mean = Float64[]
            correlations_mean_y = Float64[]
            correlations_std_x = Float64[]
            correlations_std_y = Float64[]

            for i in 1:length(lags)
                lag_mean = correlations[:, i]
                lag_mean_y = correlations_y[:, i]
                push!(correlations_mean, mean(lag_mean))
                push!(correlations_std_x, std(lag_mean))
                push!(correlations_mean_y, mean(lag_mean_y))
                push!(correlations_std_y, std(lag_mean_y))
            end


            logx_std = log.(correlations_std_x)
            logx = log.(abs.(correlations_mean))

            logy_std = log.(correlations_std_y)
            logy = log.(abs.(correlations_mean_y))

            time_005_x = findall(correlations_mean .<= 0.1)[1]
            time_005_y = findall(correlations_mean_y .<= 0.1)[1]
        
            slope_x = linear_fit_mx(lags[1:time_005_x], logx[1:time_005_x])
            slope_y = linear_fit_mx(lags[1:time_005_y], logx[1:time_005_y])

            k_x = -slope_x 
            k_y = -slope_y

            k = (k_x + k_y) / 2
            D = variance * k

            # =========================
            # STORE METRICS
            # # =========================
            push!(metrics, (
                tm ,
                v,
                l,
                sigma,
                variance,
                k,
                D,
                P_esc
            ))
        end
    end
end

In [ ]:
@save "metrics.jld2" metrics